# Assignment 2 — Zero-Shot Image Classification with Transformers

**U of T DSI — Deep Learning**  
Based on the official `02_activities/assignments/assignment_2.ipynb` assignment. The original six tasks and their order are preserved. The exercise code, observations, and report have been completed in this **adapted, self-contained notebook**.

This assignment uses an existing **OpenAI CLIP ViT-B/32** model to classify Fashion-MNIST *without training it on Fashion-MNIST*. The experiment compares bare labels with more descriptive prompts, visualizes image embeddings with UMAP, and tests multiple descriptions per class.

**AI Use Disclosure (Tier 3):** ChatGPT (OpenAI) generated code, prompt designs, experimental methodology, and explanatory prose for this notebook. Results must come from actually executing the cells. Performance values and plots have **not** been fabricated. I will review outputs and acknowledge AI use as required by the course.

**Important execution notes:** CLIP model weights and Fashion-MNIST are downloaded on first use. The full test set contains 10,000 images. CPU inference on an Intel Mac may be demanding. A temporary quick-check switch is available below, but **set `QUICK_MODE = False` for the complete assignment**. We reuse image features across all prompt experiments to avoid repeated CLIP image inference.

## 1. Loading the Fashion-MNIST Dataset

The original task uses `torchvision.datasets.FashionMNIST` on the **test** split and a `DataLoader` with `batch_size=16`, `shuffle=True`, and a `collate_clip` function. All processing is done with CLIPProcessor.

### Python environment

Run this command in VS Code Terminal **from the `deep_learning` project directory** (not inside a Python cell) if required:

```bash
uv pip install --python .venv311/bin/python "transformers>=4.44,<5" torch torchvision umap-learn scikit-learn matplotlib pandas pillow tqdm
```

Use the `.venv311` Python 3.11 notebook kernel. The project does not require TensorFlow for Assignment 2.

In [ ]:
# Environment and model setup (first run downloads pretrained CLIP weights).
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# For a brief smoke test only, set QUICK_MODE=True; switch it back to False for the assignment.
QUICK_MODE = False
QUICK_TEST_COUNT = 320

clip_model_name = "openai/clip-vit-base-patch32"
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"

clip_model = CLIPModel.from_pretrained(clip_model_name)
clip_processor = CLIPProcessor.from_pretrained(clip_model_name, use_fast=False)
clip_model.eval()
clip_model.to(device)
print("PyTorch:", torch.__version__)
print("Using device:", device)
print("CLIP model:", clip_model_name)

In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader, Subset

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

def collate_clip(batch):
    imgs, labels = zip(*batch)
    processed = clip_processor(images=list(imgs), return_tensors="pt", padding=True)
    return processed["pixel_values"].to(device), torch.tensor(labels, dtype=torch.long)

test_dataset = datasets.FashionMNIST(root=os.path.expanduser("~/.cache/uoft_dsi_fashion_mnist"), train=False, download=True)
if QUICK_MODE:
    rng = np.random.default_rng(SEED)
    selected = np.sort(rng.choice(len(test_dataset), QUICK_TEST_COUNT, replace=False))
    eval_dataset = Subset(test_dataset, selected.tolist())
else:
    eval_dataset = test_dataset

test_loader = DataLoader(
    eval_dataset, batch_size=16, shuffle=True,
    collate_fn=collate_clip, num_workers=0
)
print("Fashion-MNIST test dataset:", len(test_dataset))
print("Evaluating images:", len(eval_dataset))
print("Categories:", CLASS_NAMES)
if QUICK_MODE:
    print("QUICK MODE ONLY — results do not satisfy full-test-set reporting.")

In [ ]:
# Show the first batch. CLIP's normalized tensors are rescaled here only for display.
def show_batch(loader):
    images, labels = next(iter(loader))
    images = images.detach().cpu()
    images = (images - images.min()) / (images.max() - images.min()).clamp_min(1e-12)
    ncols = len(images)
    fig, axes = plt.subplots(1, ncols, figsize=(16, 2.5))
    for ax, img, label in zip(axes, images, labels):
        ax.imshow(img.permute(1, 2, 0))
        ax.set_title(CLASS_NAMES[int(label)], fontsize=7, rotation=35)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_batch(test_loader)

## 2. Zero-Shot Classification Baseline

Zero-shot prediction compares one **normalized image embedding** against ten **normalized text embeddings**. Their dot product equals cosine similarity. The predicted label is the column with the greatest similarity. We compute text embeddings once and **cache the image embeddings once** so the later prompt experiments require only inexpensive matrix operations.

In [ ]:
def _as_feature_tensor(output):
    """Handle tensor and feature-output wrappers across supported transformers 4.x versions."""
    if isinstance(output, torch.Tensor):
        return output
    if hasattr(output, "pooler_output") and output.pooler_output is not None:
        return output.pooler_output
    if hasattr(output, "last_hidden_state"):
        return output.last_hidden_state[:, 0]
    raise TypeError(f"Unexpected CLIP output type: {type(output)!r}")

def get_text_embeddings(prompts: list[str]) -> torch.Tensor:
    """Return L2-normalized CLIP text embeddings, shape (num_prompts, embedding_dim)."""
    tokens = clip_processor(text=prompts, padding=True, truncation=True,
                            return_tensors="pt").to(device)
    with torch.inference_mode():
        features = _as_feature_tensor(clip_model.get_text_features(**tokens))
    return features / features.norm(dim=-1, keepdim=True).clamp_min(1e-12)

def get_image_embeddings(images: torch.Tensor) -> torch.Tensor:
    """Return L2-normalized CLIP image embeddings, shape (batch, embedding_dim)."""
    with torch.inference_mode():
        features = _as_feature_tensor(clip_model.get_image_features(pixel_values=images))
    return features / features.norm(dim=-1, keepdim=True).clamp_min(1e-12)

def get_cosine_similarity(image_feats: torch.Tensor, text_feats: torch.Tensor) -> np.ndarray:
    """Return cosine similarity matrix (n_images, n_text_prompts)."""
    return (image_feats.detach().cpu() @ text_feats.detach().cpu().T).numpy()

def get_predictions(similarity: np.ndarray) -> np.ndarray:
    """Return the index of each image's highest-scoring prompt."""
    return np.argmax(similarity, axis=1)

# Small independently verifiable tests for the completed helper functions.
a = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
b = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
assert get_cosine_similarity(a, b).shape == (2, 2)
assert np.array_equal(get_predictions(get_cosine_similarity(a, b)), [0, 1])
print("Cosine similarity + prediction helper tests: PASS")

In [ ]:
# Collect *all* test-set embeddings once. Reuse these vectors in all later experiments.
all_img_emb = []
all_labels = []

for batch_index, (pixel_values, labels) in enumerate(test_loader, start=1):
    image_features = get_image_embeddings(pixel_values)
    all_img_emb.append(image_features.cpu())
    all_labels.append(labels.cpu())
    if batch_index % 50 == 0 or batch_index == len(test_loader):
        print(f"Encoded batch {batch_index}/{len(test_loader)}")

all_img_emb = torch.cat(all_img_emb, dim=0)
all_labels = torch.cat(all_labels, dim=0)
y_true = all_labels.numpy()
assert len(all_img_emb) == len(eval_dataset) == len(y_true)
assert torch.isfinite(all_img_emb).all()
print("Encoded image features:", tuple(all_img_emb.shape))
print("Ground-truth labels:", y_true.shape)

In [ ]:
# Zero-shot baseline using the official bare Fashion-MNIST class labels.
baseline_text_feats = get_text_embeddings(CLASS_NAMES)
baseline_similarity = get_cosine_similarity(all_img_emb, baseline_text_feats)
baseline_pred = get_predictions(baseline_similarity)
baseline_accuracy = accuracy_score(y_true, baseline_pred)

print(f"Bare-label zero-shot baseline accuracy: {baseline_accuracy:.2%}")
print("Confusion matrix:")
print(confusion_matrix(y_true, baseline_pred, labels=np.arange(10)))

def plot_confusion_matrix(y_true, y_pred, class_names, title="Confusion matrix"):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False, values_format="d"
    )
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return cm

baseline_cm = plot_confusion_matrix(
    y_true, baseline_pred, CLASS_NAMES, "Baseline: bare-label prompts"
)

### Reflection — Zero-shot baseline versus Assignment 1 CNN

A Fashion-MNIST CNN trained in Assignment 1 adjusts its weights using examples from these ten categories. By contrast, **CLIP receives no Fashion-MNIST-specific training here**; it relies on prior image–text pretraining and a label description. This makes zero-shot deployment flexible and fast to adapt to new classes, but the label wording is important and CLIP's prior data differs from low-resolution grayscale clothing images. Fine distinctions among *coat*, *pullover*, *shirt*, and *T-shirt/top* may be hard.

The zero-shot baseline accuracy is **computed above**, not assumed. The numerical CNN reference below comes from the **executed outputs already saved** in your GitHub Assignment 1 notebook (`assignment-1` branch): baseline linear model **83.08%**, reference CNN **89.37%**, and final selected CNN **89.63%**. We compare zero-shot CLIP against the actual **89.63%** final CNN test score, evaluated on the 10,000 Fashion-MNIST test images.

In [ ]:
# Sourced from your already executed Assignment 1 notebook on GitHub:
# https://github.com/lvsburger/deep_learning/blob/assignment-1/02_activities/assignments/assignment_1.ipynb
# Final: accuracy=89.63%; baseline=83.08%; reference CNN=89.37%.
# These are measured Assignment 1 scores (not predictions for Assignment 2).
A1_CNN_TEST_ACCURACY = 0.8963

if A1_CNN_TEST_ACCURACY is None:
    print("Assignment 1 CNN test accuracy is not entered; cannot claim a numeric comparison.")
    print(f"Measured CLIP baseline accuracy: {baseline_accuracy:.2%}")
else:
    assert 0.0 <= A1_CNN_TEST_ACCURACY <= 1.0
    gap = (baseline_accuracy - A1_CNN_TEST_ACCURACY) * 100
    print(f"Assignment 1 CNN: {A1_CNN_TEST_ACCURACY:.2%}")
    print(f"CLIP baseline:    {baseline_accuracy:.2%}")
    print(f"CLIP minus CNN:   {gap:+.2f} percentage points")

## 3. Improving Zero-Shot Classification with Prompt Engineering

The original prompts are single class names. The improved prompts give each class more *visual context*. We use exactly **one prompt per class**, preserving the one-to-one class mapping for the controlled baseline comparison.

In [ ]:
IMPROVED_PROMPTS = [
    "a grayscale catalog photo of a short-sleeved T-shirt or top",  # T-shirt/top
    "a grayscale product photo of a pair of long trousers or pants",  # Trouser
    "a grayscale product photo of a pullover sweater with long sleeves",  # Pullover
    "a grayscale product photo of a one-piece dress",  # Dress
    "a grayscale product photo of a coat or outerwear jacket",  # Coat
    "a grayscale product photo of an open-toe sandal",  # Sandal
    "a grayscale product photo of a collared shirt",  # Shirt
    "a grayscale product photo of a running sneaker or athletic shoe",  # Sneaker
    "a grayscale product photo of a handbag or shoulder bag",  # Bag
    "a grayscale product photo of an ankle-height boot",  # Ankle boot
]
assert len(IMPROVED_PROMPTS) == len(CLASS_NAMES) == 10

improved_text_feats = get_text_embeddings(IMPROVED_PROMPTS)
improved_similarity = get_cosine_similarity(all_img_emb, improved_text_feats)
improved_pred = get_predictions(improved_similarity)
improved_accuracy = accuracy_score(y_true, improved_pred)

print(f"Baseline accuracy:        {baseline_accuracy:.2%}")
print(f"Improved prompt accuracy:{improved_accuracy: .2%}")
print(f"Change in percentage points: {(improved_accuracy-baseline_accuracy)*100:+.2f}")
improved_cm = plot_confusion_matrix(
    y_true, improved_pred, CLASS_NAMES, "Improved: descriptive prompts"
)

In [ ]:
# Identify which classes gained or lost recognition, using actual results.
from sklearn.metrics import classification_report
baseline_report = classification_report(y_true, baseline_pred, labels=np.arange(10),
                                        target_names=CLASS_NAMES, output_dict=True,
                                        zero_division=0)
improved_report = classification_report(y_true, improved_pred, labels=np.arange(10),
                                        target_names=CLASS_NAMES, output_dict=True,
                                        zero_division=0)
class_comparison = pd.DataFrame({
    "class": CLASS_NAMES,
    "baseline_recall": [baseline_report[n]["recall"] for n in CLASS_NAMES],
    "improved_recall": [improved_report[n]["recall"] for n in CLASS_NAMES],
})
class_comparison["change"] = (class_comparison["improved_recall"]
                               - class_comparison["baseline_recall"])
print("Per-class recall changes:")
display(class_comparison.sort_values("change", ascending=False).round(3))
print("Summary:")
if improved_accuracy > baseline_accuracy:
    print("The descriptive prompts improved the measured overall accuracy.")
elif improved_accuracy < baseline_accuracy:
    print("The descriptive prompts reduced the measured overall accuracy.")
else:
    print("The descriptive prompts did not change the measured overall accuracy.")

### Reflection — Prompt wording and its limitations

Giving CLIP short descriptions can alter the text vectors and therefore alter image rankings, despite **no changes to the vision model weights**. Descriptions such as *short-sleeved T-shirt* versus *collared shirt*, and *open-toe sandal* versus *ankle-height boot*, were chosen to distinguish visually similar categories.

Whether the prompts actually help is determined by the **measured accuracies and per-class recall table above**. An increase in overall accuracy does not mean every class improved. If results are mixed, it suggests that descriptive prompts can be sensitive to wording, background assumptions, and whether product images resemble CLIP's original training data. Additional templates or several prompts per class may produce more reliable comparisons, which we test below.

## 4. Visualizing Image Embeddings with UMAP

Use **all cached image embeddings** from the evaluated test set. UMAP projects them from CLIP's feature dimension to 2D. Colours indicate *ground truth* categories. Neighbourhood relationships in the plot are suggestive, but the 2D distances do not preserve all properties of the original high-dimensional embedding space.

In [ ]:
from umap import UMAP

embedding_array = all_img_emb.detach().cpu().numpy()
reducer = UMAP(n_components=2, n_neighbors=20, min_dist=0.12,
               metric="cosine", random_state=42)
embedding_2d = reducer.fit_transform(embedding_array)
assert embedding_2d.shape == (len(y_true), 2)

fig, ax = plt.subplots(figsize=(10, 7))
for class_idx, class_name in enumerate(CLASS_NAMES):
    mask = (y_true == class_idx)
    ax.scatter(embedding_2d[mask, 0], embedding_2d[mask, 1],
               s=9, alpha=0.55, label=class_name)
ax.set_title("UMAP: Fashion-MNIST CLIP image embeddings (true labels)")
ax.set_xlabel("UMAP dimension 1")
ax.set_ylabel("UMAP dimension 2")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
plt.tight_layout()
plt.show()

### Reflection — UMAP separability

Because CLIP was pre-trained to align images and natural-language descriptions, some garments may form distinguishable clusters before any Fashion-MNIST-specific training. Similar shapes can overlap; for example, **shirt/pullover/coat/top** or **sneaker/sandal/ankle boot**, depending on the actual embedding distribution. Examine the plotted clusters above rather than assuming that any specific class will be perfectly separated. UMAP is an exploratory view, **not** a direct measure of classification accuracy.

## 5. Mini-Experiment — B. Multiple-Description Classification

**Chosen official option: B (Multiple-Description Classification).**

For each of the ten classes, supply **three alternative natural-language prompts**. Compute their CLIP text embeddings. The basic official decision rule chooses the **single highest-scoring prompt out of all 30** and maps it back to its parent Fashion-MNIST class; therefore any of the class's three descriptions counts as a correct class prediction.

As an additional controlled check, we also average the three prompt similarities per class. This helps us see whether a class improves by having one particularly well-matched prompt or by having several consistently good ones. All methods reuse the same test images and cached image embeddings.

In [ ]:
MULTI_PROMPTS = [
    ["a photo of a T-shirt", "a short-sleeved casual top", "a grayscale clothing catalog image of a T-shirt"],
    ["a pair of long pants", "a photo of trousers", "a grayscale clothing catalog image of trousers"],
    ["a knitted pullover sweater", "a photo of a pullover", "a grayscale clothing catalog image of a sweater"],
    ["a photo of a dress", "a one-piece outfit with a skirt", "a grayscale clothing catalog image of a dress"],
    ["a warm outerwear coat", "a photo of a long jacket", "a grayscale clothing catalog image of a coat"],
    ["an open-toe sandal", "a summer sandal shoe", "a grayscale footwear catalog image of sandals"],
    ["a photo of a shirt", "a collared button-up shirt", "a grayscale clothing catalog image of a shirt"],
    ["a pair of athletic sneakers", "a running shoe", "a grayscale footwear catalog image of a sneaker"],
    ["a photo of a handbag", "a bag with handles", "a grayscale accessory catalog image of a purse"],
    ["a photo of an ankle boot", "a closed-toe leather boot", "a grayscale footwear catalog image of ankle boots"],
]
assert len(MULTI_PROMPTS) == len(CLASS_NAMES)
assert all(len(prompts) == 3 for prompts in MULTI_PROMPTS)

flat_prompts = [prompt for prompt_list in MULTI_PROMPTS for prompt in prompt_list]
prompt_to_class = np.repeat(np.arange(len(CLASS_NAMES)), 3)
all_prompt_features = get_text_embeddings(flat_prompts)
all_prompt_similarities = get_cosine_similarity(all_img_emb, all_prompt_features)

# Official Option B approach: choose any of 30 prompts; map winning prompt -> original class.
winning_prompt_index = np.argmax(all_prompt_similarities, axis=1)
multi_pred = prompt_to_class[winning_prompt_index]
multi_accuracy = accuracy_score(y_true, multi_pred)

# Bonus: average the three similarities for each class before comparing classes.
mean_class_similarities = all_prompt_similarities.reshape(-1, 10, 3).mean(axis=2)
mean_prompt_pred = mean_class_similarities.argmax(axis=1)
mean_prompt_accuracy = accuracy_score(y_true, mean_prompt_pred)

experiment_results = pd.DataFrame([
    {"Method": "Bare class names", "Accuracy": baseline_accuracy},
    {"Method": "One descriptive prompt per class", "Accuracy": improved_accuracy},
    {"Method": "Three prompts per class (best prompt wins)", "Accuracy": multi_accuracy},
    {"Method": "Three prompts per class (mean similarity)", "Accuracy": mean_prompt_accuracy},
])
display(experiment_results.style.format({"Accuracy": "{:.2%}"}))
print("Best-prompt method versus baseline:",
      f"{(multi_accuracy-baseline_accuracy)*100:+.2f} percentage points")
print("Mean-prompt method versus baseline:",
      f"{(mean_prompt_accuracy-baseline_accuracy)*100:+.2f} percentage points")

multi_cm = plot_confusion_matrix(
    y_true, multi_pred, CLASS_NAMES,
    "Mini-experiment: three descriptions per class (best prompt)"
)

In [ ]:
# Which types of clothing were most affected by this mini-experiment?
from sklearn.metrics import recall_score
per_class_baseline = recall_score(y_true, baseline_pred,
                                  labels=np.arange(10), average=None, zero_division=0)
per_class_multi = recall_score(y_true, multi_pred,
                               labels=np.arange(10), average=None, zero_division=0)
per_class_change = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Baseline recall": per_class_baseline,
    "Multi-description recall": per_class_multi,
    "Change": per_class_multi - per_class_baseline,
})
display(per_class_change.sort_values("Change", ascending=False).round(3))

# Optional evidence-based example: first image where the baseline is wrong and multi-prompt is right.
fixed = np.flatnonzero((baseline_pred != y_true) & (multi_pred == y_true))
if len(fixed):
    i = int(fixed[0])
    print("Example corrected by multiple descriptions:")
    print("True class:", CLASS_NAMES[y_true[i]])
    print("Baseline prediction:", CLASS_NAMES[baseline_pred[i]])
    print("Multiple-description prediction:", CLASS_NAMES[multi_pred[i]])
else:
    print("No sampled image was fixed by this method; no example will be invented.")

## 6. Short Report

The following cell produces a concise, **numeric report using the measured outputs**. It includes the assignment's requested Introduction, Methodology, Results and Discussion. The introductory explanations do not depend on external accuracy estimates, and the performance sentences are generated only after the cells have been executed.

In [ ]:
from IPython.display import Markdown, display

report_lines = [
    "# Assignment 2 — Mini-Experiment Report",
    "## Introduction",
    "I evaluated zero-shot Fashion-MNIST classification using a pretrained CLIP vision–language model. "
    "My mini-experiment tested whether giving CLIP three alternative descriptions per clothing class "
    "changes its predictions, without fine-tuning or training on Fashion-MNIST labels.",
    "## Methodology",
    f"I used `{clip_model_name}` with torchvision's Fashion-MNIST test data. "
    f"The evaluation contained **{len(y_true):,}** images. Every image was encoded once; "
    "image embeddings were L2-normalized and compared with L2-normalized prompt embeddings "
    "using cosine similarity. The baseline used ten bare class names; the descriptive test used one "
    "sentence per class. The mini-experiment used three descriptions per class (30 total), "
    "with a class prediction obtained by mapping the highest-scoring prompt back to its class. "
    "I also compared averaging similarities within each class. UMAP visualized the frozen image embeddings.",
    "## Results",
    f"- Bare-label baseline accuracy: **{baseline_accuracy:.2%}**.",
    f"- One descriptive prompt per class: **{improved_accuracy:.2%}** "
    f"({(improved_accuracy-baseline_accuracy)*100:+.2f} percentage points versus baseline).",
    f"- Three prompts, highest-scoring description: **{multi_accuracy:.2%}** "
    f"({(multi_accuracy-baseline_accuracy)*100:+.2f} percentage points versus baseline).",
    f"- Three prompts, mean similarity by class: **{mean_prompt_accuracy:.2%}** "
    f"({(mean_prompt_accuracy-baseline_accuracy)*100:+.2f} percentage points versus baseline).",
    "The confusion matrices and per-class recall table above provide the breakdown of errors.",
    "## Discussion",
    ("Multiple descriptions **improved** the overall top-1 accuracy compared with bare class names. "
      if multi_accuracy > baseline_accuracy else
     "Multiple descriptions **did not improve** overall top-1 accuracy compared with bare class names. "),
    "This demonstrates that CLIP's zero-shot predictions depend on the natural-language prompts, "
    "even though its weights remain unchanged. Additional prompts can help describe alternative "
    "visual appearances, but can also introduce ambiguous wording and distract the classifier. "
    "UMAP gives a qualitative view of class overlap, not a direct accuracy score. "
    f"Compared with the final CNN already evaluated in Assignment 1 (**{A1_CNN_TEST_ACCURACY:.2%}**), "
    f"the measured CLIP baseline differs by **{(baseline_accuracy-A1_CNN_TEST_ACCURACY)*100:+.2f} percentage points**. "
    "CLIP requires no Fashion-MNIST-specific training and can accommodate new class descriptions, "
    "but its predictions depend on prompt choice and transfer from web-like images to low-resolution "
    "grayscale clothes. The CNN reference figure comes from the user's previously saved, executed "
    "Assignment 1 GitHub notebook; all CLIP figures come from running this notebook.",
    "## AI Use Disclosure",
    "I used ChatGPT (OpenAI) extensively to generate the implementation, prompt designs, "
    "experiment and this report structure. Measured results were generated by running this notebook; "
    "the AI did not supply invented accuracy values.",
]
if QUICK_MODE:
    report_lines.insert(1, "**QUICK CHECK ONLY: this run used a reduced test subset and is not full-assignment reporting.**")
report_md = "\n\n".join(report_lines)
display(Markdown(report_md))

### Submission notes and checklist

- [x] Code supplied for the Fashion-MNIST test DataLoader and CLIP preprocessing.
- [x] Baseline cosine similarity, prediction, measured accuracy, and confusion matrix.
- [x] More descriptive class prompts, measurement, and reflection.
- [x] UMAP on the cached evaluation embeddings.
- [x] Option B mini-experiment with three descriptions per class and an optional mean-similarity comparison.
- [x] Executable short report with measured figures and AI disclosure.
- [ ] **You must run all cells and inspect the actual outputs.** This file was built and syntax-checked, but CLIP inference and UMAP were not executed during creation.
- [ ] For final submission, ensure `QUICK_MODE = False` and report uses 10,000 test images.
- [x] Assignment 1 CNN comparison uses your real, previously executed final CNN test result of **89.63%**, verified from your `assignment-1` GitHub Notebook.

**Official assignment deadline:** 2026-11-17 at 23:59 (Toronto time).  
**Submission branch:** `assignment-2` (not `assignment-1`).  
**Submit these four files:** `02_activities/assignments/assignment_2.ipynb`, `01_materials/labs/lab_4.ipynb`, `lab_5.ipynb`, and `lab_6.ipynb` at the same labs path.

Create a PR from `assignment-2` to `main`, check it is accessible, and **do not merge the PR yourself**. If local files already exist, back them up before replacing them with downloaded versions. The saved `.ipynb` outputs must be included if requested for assessment.